# Ценовой блок ДОМ.РФ + сделки/аренда

Источник: `sandbox_datasets/dom-price-deals-20260921.xlsx` (VPS → ВМ, 2026-09-21).
15 метрик, 30 224 наблюдения. Листы: **Метаданные** (с колонкой «ЕДИНИЦА — ваш ввод»), **Данные (long)**, **Max регионы + RF**, **Сводка RF квартал**.

Контекст: каталог показателей (этап 3), единицы измерения заполняет владелец в XLS; после заполнения значения возвращаются в `derived.catalog_facets.unit_canonical`.

In [1]:
import pandas as pd

XLS = '/home/ubuntu/sandbox_datasets/dom-price-deals-20260921.xlsx'
meta = pd.read_excel(XLS, sheet_name='Метаданные')
long = pd.read_excel(XLS, sheet_name='Данные (long)')
maxrf = pd.read_excel(XLS, sheet_name='Max регионы + RF')
rfq = pd.read_excel(XLS, sheet_name='Сводка RF квартал')
print('метрик:', len(meta), '| наблюдений:', len(long))
meta.head(16)

метрик: 15 | наблюдений: 30224


,metric_id,metric_code,Название,Частота,unit_id,Свежесть,Глубина (точек),Последнее наблюдение,Домен,Мера,ЕДИНИЦА (ваш ввод),Комментарий
0,1142,itzdr,Индекс цен на жильё ДОМ.РФ,кварт,18,fresh,5025,2026-07-01,housing,index,NaN,NaN
1,1143,itzdrpm,"Индекс цен на жильё ДОМ.РФ, % к предыдущему ме...",кварт,18,fresh,5025,2026-07-01,housing,growth_rate,NaN,NaN
2,1144,csdr,"Число сделок, ДОМ.РФ",кварт,18,fresh,5025,2026-07-01,credit,level,NaN,NaN
3,1159,ksdrk,"Количество сделок ДКП, регион, квартал",кварт,18,fresh,854,2026-04-01,other,level,NaN,NaN
4,1161,mtdrk,"Медианная цена ДКП за м², регион, квартал",кварт,18,fresh,854,2026-04-01,prices,level,NaN,NaN
5,1160,mtsrk,"Медианная цена сделки за м², регион, квартал",кварт,18,fresh,854,2026-04-01,prices,level,NaN,NaN
6,1162,mtdrk_2,"Медианная цена ДДУ за м², регион, квартал",кварт,18,fresh,840,2026-04-01,housing,level,NaN,NaN
7,1164,msark,"Медианная ставка аренды за м², регион, квартал",кварт,18,fresh,833,2026-04-01,credit,level,NaN,NaN
8,1165,mpazrk,"Медианная площадь арендуемого жилья, регион, к...",кварт,18,fresh,833,2026-04-01,housing,level,NaN,NaN
9,1163,kdark,"Количество договоров аренды, регион, квартал",кварт,18,fresh,833,2026-04-01,other,level,NaN,NaN


## Метаданные — заполните колонку «ЕДИНИЦА (ваш ввод)»
unit_id=18 означает «не определена в БД» — ваша колонка закроет пробел каталога.

In [2]:
meta[['metric_code','Название','Частота','Свежесть','Глубина (точек)','ЕДИНИЦА (ваш ввод)']]

,metric_code,Название,Частота,Свежесть,Глубина (точек),ЕДИНИЦА (ваш ввод)
0,itzdr,Индекс цен на жильё ДОМ.РФ,кварт,fresh,5025,NaN
1,itzdrpm,"Индекс цен на жильё ДОМ.РФ, % к предыдущему ме...",кварт,fresh,5025,NaN
2,csdr,"Число сделок, ДОМ.РФ",кварт,fresh,5025,NaN
3,ksdrk,"Количество сделок ДКП, регион, квартал",кварт,fresh,854,NaN
4,mtdrk,"Медианная цена ДКП за м², регион, квартал",кварт,fresh,854,NaN
5,mtsrk,"Медианная цена сделки за м², регион, квартал",кварт,fresh,854,NaN
6,mtdrk_2,"Медианная цена ДДУ за м², регион, квартал",кварт,fresh,840,NaN
7,msark,"Медианная ставка аренды за м², регион, квартал",кварт,fresh,833,NaN
8,mpazrk,"Медианная площадь арендуемого жилья, регион, к...",кварт,fresh,833,NaN
9,kdark,"Количество договоров аренды, регион, квартал",кварт,fresh,833,NaN


## Сводка РФ по кварталам
Сделки ДКП, договоры аренды, медианная цена ДДУ (РФ). Обратите внимание: медианная цена ДДУ скачет по кварталам — вероятно, меняется состав регионов.

In [3]:
rfq.sort_values('period')

,period,ksdrk_2 (сделки ДКП РФ),kdark_2 (аренда РФ),mtdrk_2 (мед.цена ДДУ РФ)
0,2024-01-01,701618,55211,58704.7
1,2024-04-01,873033,58034,29818.1
2,2024-07-01,659285,48430,20330.8
3,2024-10-01,687328,54325,35150.0
4,2025-01-01,535272,49742,651.9
5,2025-04-01,630773,46033,165000.0
6,2025-07-01,703116,46064,74573.9
7,2025-10-01,768789,53232,395176.5
8,2026-01-01,561483,36844,121865.4
9,2026-04-01,733588,37458,92803.3


## Max по регионам + RF
На каждый период: максимум среди регионов (region_id>1) и имя региона-лидера; справа — РФ-факты из отдельных агрегатных рядов (ksdrk_2, kdark_2).

In [4]:
maxrf.fillna('')

,период,itzdr MAX,itzdr регион-макс,itzdrpm MAX,itzdrpm регион-макс,csdr MAX,csdr регион-макс,ksdrk MAX,ksdrk регион-макс,mtsrk MAX,...,mtdrk MAX,mtdrk регион-макс,mtdrk_2 MAX,mtdrk_2 регион-макс,kdark MAX,kdark регион-макс,mpazrk MAX,mpazrk регион-макс,ksdrk_2 РФ (сделки ДКП),kdark_2 РФ (аренда)
0,2021-01-01,100.00000,Белгородская область,0.000000,Белгородская область,4833,Москва,,,,...,,,,,,,,,,
1,2021-02-01,105.95639,Сахалинская область,5.956389,Сахалинская область,6543,Москва,,,,...,,,,,,,,,,
2,2021-03-01,106.80637,Краснодарский край,6.806375,Краснодарский край,7391,Москва,,,,...,,,,,,,,,,
3,2021-04-01,114.18255,Республика Адыгея,14.182547,Республика Адыгея,7981,Москва,,,,...,,,,,,,,,,
4,2021-05-01,126.59021,Республика Адыгея,26.590208,Республика Адыгея,6389,Москва,,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,2026-03-01,271.45720,Республика Адыгея,11.615402,Курская область,3538,Москва,,,,...,,,,,,,,,,
63,2026-04-01,273.07785,Кировская область,11.779202,Курская область,3403,Москва,41499.0,Московская область,327301.0,...,326167.1,Москва,410039.8,Москва,1675.0,Республика Башкортостан,700000.0,Республика Тыва,733588.0,37458.0
64,2026-05-01,274.85410,Кировская область,17.220154,Тверская область,3303,Москва,,,,...,,,,,,,,,,
65,2026-06-01,281.36844,Кировская область,17.116537,Тверская область,4575,Москва,,,,...,,,,,,,,,,


## Динамика медианных цен по регионам
Разворот long → pivot по периодам для ценовых рядов.

In [5]:
# Медианные цены по регионам (пример: mtsrk - медианная цена сделки за м2)
import psycopg
con = psycopg.connect('host=localhost port=5432 dbname=research_wiki user=wiki')
df = pd.read_sql('''SELECT o.period_start::date p, r.name_ru region, o.value
FROM core.observation_v2 o JOIN core.region r USING(region_id)
WHERE o.metric_id=1160 AND o.region_id>1 ORDER BY o.period_start, o.value DESC''', con)
pivot = df.pivot_table(index='p', columns='region', values='value', aggfunc='first')
pivot.iloc[:, :12]  # топ-12 регионов по объёму на первый период

/tmp/ipykernel_445862/1422301521.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('''SELECT o.period_start::date p, r.name_ru region, o.value


region,Алтайский край,Амурская область,Архангельская область (без НАО),Астраханская область,Белгородская область,Брянская область,Владимирская область,Волгоградская область,Вологодская область,Воронежская область,Еврейская автономная область,Забайкальский край
p,,,,,,,,,,,,
2024-01-01,9175.7,12263.1,34893.9,11056.9,6000.0,7463.5,7552.9,10887.7,12306.2,10359.1,11597.9,7785.5
2024-04-01,7704.9,12633.4,25076.0,9881.4,4865.5,6106.2,5000.0,10000.0,5537.6,8333.3,8647.0,7604.6
2024-07-01,12072.2,16811.9,28882.3,14798.4,8695.7,8689.8,8992.8,12890.3,8750.0,12430.9,14403.3,11435.1
2024-10-01,12468.8,19798.8,30000.0,14073.5,10000.0,11297.3,10056.7,13611.3,18102.7,13961.0,17449.7,12681.2
2025-01-01,10558.1,13274.3,20190.0,8333.3,9090.9,7490.6,7100.2,9009.0,2252.4,8660.5,6494.8,7437.7
2025-04-01,10142.3,16364.6,24834.4,11159.8,7884.7,8707.1,7272.7,10373.4,5956.3,10580.5,15112.4,9080.6
2025-07-01,11041.0,16891.9,27388.5,10526.3,8591.1,9057.1,7692.3,12839.0,5049.0,10352.0,9432.6,9708.7
2025-10-01,14285.7,19768.8,36885.8,14977.0,9788.4,11603.3,11834.3,14008.1,6147.5,14909.4,16875.2,14613.8
2026-01-01,17889.1,22554.3,43252.6,16200.6,11072.8,11262.3,17764.2,16867.6,4747.1,19078.8,20574.5,15089.5


## После заполнения единиц
Экспортируйте лист «Метаданные» обратно в `~/sandbox_datasets/` — агент заберёт файл, проверит колонку и зальёт единицы в `derived.catalog_facets.unit_canonical` + вернёт XLS в репозиторий.